# Martini, Molano-Mazon and Maravall (2017) on CRCNS ssc-6

Paper: Martini FJ, Molano-Mazon M, Maravall M (2017). Interspersed Distribution of Selectivity to
Kinematic Stimulus Features in Supragranular Layers of Mouse Barrel Cortex. Cerebral Cortex
27:3782-3789, doi:10.1093/cercor/bhx019.

Dataset: CRCNS ssc-6 (doi:10.6080/K02F7KCP), the one file `ssc-6/live/data.tar.gz` (8,062,238,498
bytes, MD5 `eb7382e21fe469614d288923ae760f7a` in the data set's `checksums.md5`), streamed from the
public S3 bucket `crcnsarchive`. CRCNS data sets are not versioned, so the version is recorded as
`unversioned` with the data set's DOI and the MD5 of the file read, which the notebook checks
against the live `checksums.md5`; the archive holds no other version of this data set (`ssc-6/`
has only `live/`), so there was nothing else to choose between. The data description read is
`ssc-6/live/docs/crcns_ssc-6_data_description.pdf` (version 0.7, April 10, 2017; MD5
`4180dcbcf3513a736c05d347ec31b638`).

Claims file: `claims/martini-2017.yaml`, written from the version of record (main text, Methods,
legends and the supplement's figure legends). The data set holds raw movies and stimulus files
only: no ROIs, traces or tuning labels. Every measure here is computed from the raw movies with
the Methods' pipeline as far as it is stated (`ssc6.py` says where each parameter comes from); the
ROIs are found again by an automatic detector, so neuron identities and counts differ from the
paper's.

| Claim | Tested here |
|---|---|
| dataset-size | tested |
| stimulus-set | tested |
| fig2ab-example-neuron | blocked: the example neuron is not identified |
| responsive-fraction | tested |
| fig2de-tuned-fraction | tested |
| fig2e-all-categories | tested |
| tuning-consistency | tested |
| shuffle-significance | tested (the barrel-map parts cannot be) |
| fig3b-example-correlation | blocked: the example field is not identified |
| fig3c-correlation-above-shuffle | tested |
| neuropil-correlation | tested |
| fig3d-example-correlation-distance | blocked: depends on fig3b |
| fig3e-correlation-falls-with-distance | tested |
| correlation-gradient-beyond-200um | tested |
| fig3f-example-si-distance | blocked: depends on fig3b |
| fig3g-si-no-spatial-organization | tested |
| fig4c-barrel-septum-tuned | blocked: fields cannot be placed on the barrel map |
| fig4d-border-distance-by-feature | blocked: depends on fig4c |

In [ ]:
import collections, io, itertools, resource, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ssc6

DATASET = "crcns:ssc-6/unversioned"
t_start = time.time()
checks = urllib.request.urlopen(ssc6.URL.rsplit("/", 1)[0] + "/checksums.md5").read().decode()
print(checks)
assert "eb7382e21fe469614d288923ae760f7a  data.tar.gz" in checks
data = ssc6.stream(log=lambda s: None)
for s in data["stims"].values():
    s["level"], s["norm"] = ssc6.amplitude_levels(s)
keys = sorted(data["fields"])
print(f"streamed and reduced in {(time.time() - t_start) / 60:.1f} min; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

What was loaded, against the data description (`paper/ssc-6.data-description.txt`): ten mouse
folders, "fields acquired per mouse ranged from 3 to 6", four repetitions per field, one stimulus
file per movie, movies of 833 frames of 128 x 512 pixels at 8 bits, 114 ms per frame and 1.48
pixel/um, stimulus files of 950000 x 3 samples at 10 kHz (time, piezo command in V, frame pulses
in V), and histology for experiments 8 to 10.

In [ ]:
members = [n for n, _ in data["members"]]
print("tar members:", len(members), "| files by kind:",
      dict(collections.Counter(n.rsplit(".", 1)[-1] for n, sz in data["members"] if "." in n.rsplit("/", 1)[-1])))
print("README.txt in the tar:", data["readme"]["data/README.txt"].splitlines()[0])
by_mouse = collections.defaultdict(list)
for k in keys:
    by_mouse[k[0]].append(k[1])
print("\nmouse: field ids (fields)")
for m in sorted(by_mouse):
    print(f"  exp{m}: {by_mouse[m]} ({len(by_mouse[m])})")
print("mice", len(by_mouse), "| fields", len(keys), "| movies", sum(len(data['fields'][k]['reps']) for k in keys))
print("fields per mouse range", min(map(len, by_mouse.values())), "to", max(map(len, by_mouse.values())))
f0 = data["fields"][keys[0]]
print("movie frames per recording:", sorted({data['fields'][k]['traces'][r]['F'].shape[1] for k in keys for r in data['fields'][k]['reps']}),
      "| image shape:", sorted({data['fields'][k]['mean_img'].shape for k in keys}))
missing = [(k[0], k[1], r) for k in keys for r in data["fields"][k]["reps"] if (k[0], k[1], r) not in data["stims"]]
print("movies without a stimulus file (mouse, field, repetition):", missing)
print("stimulus files without a movie:", [s for s in data["stims"] if (s[0], s[1]) not in data["fields"]])

S = data["stims"]
print("\nstimulus files:", len(S), "| samples per file:", collections.Counter(s["n_samples"] for s in S.values()),
      "| duration s:", collections.Counter(round(s["duration_s"], 3) for s in S.values()))
print("frame pulses per file (as recorded):", collections.Counter(s["n_pulses"] for s in S.values()))
fi = np.concatenate([np.diff(s["frames"]) for s in S.values()])
print(f"frame interval {np.median(fi) * 1000:.1f} ms (range {fi.min() * 1000:.1f} to {fi.max() * 1000:.1f})")
late = [k for k, s in S.items() if s["missed_first_pulse"]]
print("files starting after the first frame pulse:", late)
phase = {k: (s["onsets"][0] - s["frames"][0]) / np.median(np.diff(s["frames"])) for k, s in S.items()}
print("first deflection onset, in frames after frame 0 (frame 0 put one period before the first pulse "
      "in those files): on-pulse files", sorted({round(v, 2) for k, v in phase.items() if k not in late}),
      "| late files", sorted({round(v, 2) for k, v in phase.items() if k in late}))
print("command resting level V:", round(min(s['baseline_v'] for s in S.values()), 3), "to", round(max(s['baseline_v'] for s in S.values()), 3))

f = data["fields"][keys[0]]
mx = max(int(np.abs(data['fields'][k]['shifts'][r]).max()) for k in keys for r in data['fields'][k]['reps'])
big = [(k, r) for k in keys for r in data['fields'][k]['reps'] if np.abs(data['fields'][k]['shifts'][r]).max() > 3]
print(f"\nregistration: largest frame shift {mx} px; recordings with any shift over 3 px: {big}")
print("shifts between the movies of a field (px):", collections.Counter(tuple(int(v) for v in data['fields'][k]['between'][r]) for k in keys for r in data['fields'][k]['reps']))
fps = collections.defaultdict(list)
for k in keys:
    for r in data["fields"][k]["reps"]:
        fps[data["fields"][k]["fingerprint"][r]].append((k, r))
print("movies deposited twice (identical first frame sum, mean and spread):", [v for v in fps.values() if len(v) > 1])

nroi = {k: len(data["fields"][k]["rois"]) for k in keys}
print("\nROIs found:", sum(nroi.values()), "| per field median", np.median(list(nroi.values())),
      "range", min(nroi.values()), "to", max(nroi.values()))
print("ROIs per mouse:", {m: sum(nroi[(m, i)] for i in by_mouse[m]) for m in sorted(by_mouse)})
areas = np.array([r["area_um2"] for k in keys for r in data["fields"][k]["rois"]])
print(f"ROI area um2: median {np.median(areas):.0f}, range {areas.min():.0f} to {areas.max():.0f}")
raw = np.concatenate([data['fields'][k]['traces'][r]['F'].ravel() for k in keys for r in data['fields'][k]['reps']])
print(f"raw ROI fluorescence (8-bit units): {raw.min():.1f} to {raw.max():.1f}")

In [ ]:
FD = {k: ssc6.field_data(data, k) for k in keys}
N = sum(len(FD[k]["R"]) for k in keys)
print("ROIs dropped because F0 reaches 0 (no dF/F0):", sum(FD[k]["n_dropped"] for k in keys), "| kept:", N)
allR = np.concatenate([FD[k]["R"].ravel() for k in keys])
print(f"response amplitudes (dF/F0 as a fraction): median {np.nanmedian(allR):.4f}, 1st-99th percentile "
      f"{np.nanpercentile(allR, 1):.3f} to {np.nanpercentile(allR, 99):.3f}, NaN {np.isnan(allR).sum()}")

# Clock check: the frames and the deflections share the stimulus file's clock. Contrast the
# population response to the largest and the smallest amplitude, with true onsets and with onsets
# moved by 0.5 s (half the inter-deflection interval).
def contrast(k, shift):
    f = data["fields"][k]; vals = {1: [], 3: []}
    for r in FD[k]["reps"]:
        st = S[(k[0], k[1], r)]
        d, _ = ssc6.dff(f["traces"][r]["corrected"])
        R = ssc6.responses(d, st["frames"], st["onsets"] + shift)
        for lv in (1, 3):
            vals[lv].append(np.nanmean(R[:, st["level"] == lv]))
    return np.mean(vals[3]) - np.mean(vals[1])
ct = np.array([[contrast(k, 0.0), contrast(k, 0.5)] for k in keys])
print("\nlargest minus smallest amplitude, population mean response (dF/F0):")
for m in sorted(by_mouse):
    sel = [i for i, k in enumerate(keys) if k[0] == m]
    print(f"  exp{m}: true onsets {ct[sel, 0].mean():+.4f}, onsets + 0.5 s {ct[sel, 1].mean():+.4f}")
print(f"fields with a larger contrast at true onsets than shifted: {(ct[:, 0] > ct[:, 1]).sum()} of {len(keys)}; "
      f"Wilcoxon p = {stats.wilcoxon(ct[:, 0], ct[:, 1]).pvalue:.2g}")
print("fields where it is not:", [k for i, k in enumerate(keys) if ct[i, 0] <= ct[i, 1]])
lt = [i for i, k in enumerate(keys) if any((k[0], k[1], r) in late for r in FD[k]["reps"])]
print("fields with a late-starting stimulus file, contrast true / shifted:", [(keys[i], round(ct[i, 0], 4), round(ct[i, 1], 4)) for i in lt])

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
k = keys[0]
ax[0].imshow(data["fields"][k]["mean_img"], cmap="gray"); ax[0].set_title(f"mean image, mouse {k[0]} field {k[1]}")
ax[1].imshow(data["fields"][k]["mean_img"], cmap="gray")
c = np.array([r["centroid_px"] for r in data["fields"][k]["rois"]])
ax[1].plot(c[:, 1], c[:, 0], "r+"); ax[1].set_title(f"{len(c)} ROIs found (red), all inside the image")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
for m, h in sorted(data["histology"].items()):
    print(f"histology exp{m}: {h['n_pages']} page(s), made with {h['creator']}, embedded images "
          f"{[p['n_images'] for p in h['pages']]}, text {[p['text'].split() for p in h['pages']]}")
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, (m, h) in zip(ax, sorted(data["histology"].items())):
    a.imshow(h["pages"][0]["render"]); a.set_title(f"exp{m} histology/figure.pdf"); a.axis("off")
plt.tight_layout(); plt.show()

The data match the description in the frame size, frame count, sampling and stimulus file layout.
They differ from it in these points, each printed above: three mice have more fields than the
stated range (up to 8); six movies, the first recording of a field in six mice, have no stimulus
file; nine stimulus files start just after a frame pulse, and in them the deflections fall one
frame earlier relative to the first pulse than in the others, so one frame is put before the
file's start (the onset then falls at the same frame phase in every file); the tar's README and the
description name the data set "ssc-5". The clock check: the response to the largest amplitude
exceeds that to the smallest at the true onsets in most fields, and the contrast falls when the
onsets are moved by half the interval, so the frames and the stimulus command share the clock, as
far as this contrast shows; fields where it does not hold are listed. The histology PDFs are
single-page Illustrator figures (rendered above); they show the barrel reconstruction, the in vivo
surface and one dashed square for the imaged area, with no coordinates of each field or of its
neurons.

## dataset-size

> Layer 2/3 neurons were imaged in 10 anesthetized mice, 49 fields of view and 1054 neurons in all;
> each stimulus was presented 10 times per recording, in 4 recordings per field.

In [ ]:
reps_per_field = collections.Counter(len(data["fields"][k]["reps"]) for k in keys)
cls_counts = collections.Counter()
per_file = []
for kk, s in S.items():
    c = collections.Counter(f"{a}{l}" for a, l in zip(s["shape"], s["level"]))
    per_file.append(c)
full = [c for c in per_file if sum(c.values()) == 89]
print("mice:", len(by_mouse), "(expected 10)")
print("fields of view:", len(keys), "(expected 49)")
print("recordings (movies) per field:", dict(reps_per_field), "(expected 4); with a stimulus file:",
      dict(collections.Counter(len(FD[k]['reps']) for k in keys)))
print("deflections per stimulus file:", dict(collections.Counter(sum(c.values()) for c in per_file)))
print("presentations per stimulus in the", len(full), "files with 89 deflections:",
      {cl: sorted({c[cl] for c in full}) for cl in sorted(set().union(*full))})
print("neurons (ROIs found):", sum(nroi.values()), "| kept after the F0 check:", N, "(expected 1054, tolerance 105)")

**Result: partially reproduced.** Mice (10), fields (49) and movies per field (4) match. Each
complete stimulus file holds 89 deflections, not 90: eight stimuli are presented 10 times and the
smallest position stimulus 9 times, so the presentation count fails its zero tolerance for that
stimulus; six movies have no stimulus file. The automatic ROI detection finds 2626 ROIs, about 2.5
times the paper's 1054, outside the 10% tolerance.

## stimulus-set

> The stimulus set holds idealized position, velocity and acceleration waveforms, each at three
> amplitudes in the ratio 1:3:5, randomly interleaved and separated by 1 s.

In [ ]:
amps = {c: np.concatenate([s["amp_v"][s["shape"] == c] for s in S.values()]) for c in "PVA"}
lev = {c: np.concatenate([s["level"][s["shape"] == c] for s in S.values()]) for c in "PVA"}
print("distinct shapes:", sorted(amps))
for c in "PVA":
    m = [amps[c][lev[c] == l].mean() for l in (1, 2, 3)]
    print(f"{c}: amplitude levels {sorted(set(lev[c]))}, mean peak V {np.round(m, 3)}, "
          f"ratio largest/smallest {m[2] / m[0]:.3f}, middle/smallest {m[1] / m[0]:.3f}, "
          f"spread within a level (SD, V) {[round(float(amps[c][lev[c] == l].std()), 4) for l in (1, 2, 3)]}")
ints = np.concatenate([np.diff(s["onsets"]) for s in S.values()])
print(f"onset intervals: median {np.median(ints):.4f} s; {(np.abs(ints - 1) >= 0.1).sum()} of {len(ints)} more than 0.1 s from 1 s "
      f"(range {ints.min():.3f} to {ints.max():.3f} s)")
seqs = collections.Counter(tuple(f"{a}{l}" for a, l in zip(s["shape"], s["level"])) for s in S.values())
top, ntop = seqs.most_common(1)[0]
print(f"distinct orders of the deflections over {len(S)} stimulus files: {len(seqs)}; "
      f"the most common is in {ntop} files")
odd = {k: len(s["onsets"]) for k, s in S.items() if tuple(f"{a}{l}" for a, l in zip(s["shape"], s["level"])) != top}
print("files with another order (deflections):", odd)
print("first 18 deflections of the common order:", " ".join(top[:18]))
# within a recording the order is mixed: how often does the class change between neighbours
print("consecutive deflections of the same class in the common order:", sum(a == b for a, b in zip(top, top[1:])), "of", len(top) - 1)

**Result: partially reproduced.** Three shapes at three amplitudes each, with ratios of 5.0 and
3.0 to the smallest, and onsets 1 s apart (median 1.0001 s), as the claim states. The order is
mixed within a recording (2 of 88 neighbours share a class), but it does not differ between
recordings: 188 of 190 stimulus files hold the same order, and the other two are incomplete files
of 52 and 87 deflections. The criterion's condition that the order differs between recordings
fails.

## fig2ab-example-neuron (blocked)

The paper does not say which mouse, field or ROI holds the example neuron (claims file, open detail
11). Figure 2C shows a crop of about ten cell masks of one field with a 25 um scale bar; the data set
holds no ROI masks, and the ROIs here are found again, so a match by mask is not possible, and a match
by eye of a ten-cell crop against 49 fields was not attempted. Blocked: metadata insufficient.

## responsive-fraction

> About half (50.2%) of all neurons in the data set responded to whisker stimulation.

Readings of the criterion: (a) mean response over all presentations above zero, one-sided
Wilcoxon signed-rank p < 0.05; (b) at least one of the nine stimuli above zero (signed-rank over its
presentations), Benjamini-Yekutieli over the neurons and stimuli of a field.

In [ ]:
ra, rb, by_m = [], [], collections.defaultdict(list)
for k in keys:
    R, sh, lv = FD[k]["R"], FD[k]["shape"], FD[k]["level"]
    ok = ~np.isnan(R[0])
    a = np.array([stats.wilcoxon(r[ok], alternative="greater").pvalue for r in R]) < 0.05
    P = np.array([[stats.wilcoxon(r[ok & (sh == s) & (lv == l)], alternative="greater").pvalue
                   for s in "PVA" for l in (1, 2, 3)] for r in R])
    b = ssc6.fdr(P, method="dep").any(1)
    b_bh = ssc6.fdr(P, method="pdep").any(1)
    ra.append(a); rb.append(b); by_m[k[0]].append((a.sum(), b.sum(), b_bh.sum(), len(a)))
ra, rb = np.concatenate(ra), np.concatenate(rb)
print(f"(a) mean response > 0: {100 * ra.mean():.1f}% of {N}")
print(f"(b) any stimulus > 0, BY: {100 * rb.mean():.1f}%")
bbh = sum(v[2] for vv in by_m.values() for v in vv)
print(f"variant (b) with BH in place of BY: {100 * bbh / N:.1f}%")
print("expected 50.2% (45.2 to 55.2)")
for m in sorted(by_m):
    a, b, _, n = np.sum(by_m[m], 0)
    print(f"  exp{m}: (a) {100 * a / n:.1f}%  (b) {100 * b / n:.1f}%  of {n}")

**Result: not reproduced.** Reading (a) gives 44.7% and reading (b) 17.5%, both below the
tolerance of 45.2 to 55.2%; (a) misses it by 0.5 points. The paper does not define a responding
neuron, so the outcome rests on these readings.

## fig2de-tuned-fraction

> Most neurons were not tuned to any kinematic feature; 71.8% of all neurons were non-tuned, so
> about 30% were tuned.

The readings of the criterion: three separate regressions or one joint regression; BH (`pdep`)
or BY (`dep`); FDR family all 3 x N p-values of a field or N per feature; significant slopes of
either sign or positive only. The main reading is the first row: separate regressions (Figure 2B
plots one line per waveform type), `pdep` (the default of fdr_bh, which the Methods name), the
family of all p-values of the field (Methods, "for all neurons in a field of view ... in the
simultaneously recorded population") and either sign (the two-sided test of the slope).

In [ ]:
fits = {k: {"sep": ssc6.regress_separate(FD[k]["R"], FD[k]["shape"], FD[k]["norm"]),
            "joint": ssc6.regress_joint(FD[k]["R"], FD[k]["shape"], FD[k]["norm"])} for k in keys}
def classes(sig):
    n = sig.sum(1)
    return np.where(n == 0, "none", np.where(n > 1, "mixed", np.array(list("PVA"))[np.argmax(sig, 1)]))
readings = list(itertools.product(["sep", "joint"], ["pdep", "dep"], ["all", "feature"], ["both", "positive"]))
TUN = {}
print("reading                          non-tuned pooled  mean over fields   P    V    A  mixed")
for rd in readings:
    reg, meth, fam, sign = rd
    sig = {k: ssc6.tuning(fits[k][reg][0], fits[k][reg][1], meth, fam, sign) for k in keys}
    TUN[rd] = sig
    allc = np.concatenate([classes(sig[k]) for k in keys])
    pooled = 100 * np.mean(allc == "none")
    fmean = 100 * np.mean([np.mean(classes(sig[k]) == "none") for k in keys])
    cc = collections.Counter(allc)
    ok = "match" if abs(pooled - 71.8) <= 7.2 or abs(fmean - 71.8) <= 7.2 else ""
    print(f"{str(rd):34s} {pooled:6.1f}%          {fmean:6.1f}%        {cc['P']:4d} {cc['V']:4d} {cc['A']:4d} {cc['mixed']:5d}  {ok}")
MAIN = ("sep", "pdep", "all", "both")
print("\nexpected 71.8% non-tuned (64.6 to 79.0)")
print("main reading by mouse (non-tuned %, neurons):")
for m in sorted(by_mouse):
    allc = np.concatenate([classes(TUN[MAIN][k]) for k in keys if k[0] == m])
    print(f"  exp{m}: {100 * np.mean(allc == 'none'):.1f}% of {len(allc)}")
nt = np.array([np.mean(classes(TUN[MAIN][k]) == "none") for k in keys])
print(f"main reading, non-tuned per field: median {100 * np.median(nt):.0f}%, range {100 * nt.min():.0f} to {100 * nt.max():.0f}%; "
      f"fields with no tuned neuron: {(nt == 1).sum()}")

**Result: reproduced.** Under the main reading 78.4% of neurons are non-tuned (pooled and as the
mean over fields), within 64.6 to 79.0%, and non-tuned neurons are the majority under every
reading. Ten of the 16 readings fall within the tolerance; the other six give 64.5% (joint
regression, BH, per feature, either sign) or 79.1 to 87.3% (every BY reading with separate
regressions, and BH over the field with positive slopes only). The mice differ widely, from 39.3%
to 99.5% non-tuned.

## fig2e-all-categories

> Among tuned neurons all categories of selectivity are represented, neurons tuned to position
> only, velocity only, acceleration only, and neurons with mixed selectivity.

In [ ]:
for rd in readings:
    allc = np.concatenate([classes(TUN[rd][k]) for k in keys])
    cc = collections.Counter(allc)
    print(f"{str(rd):34s} P {cc['P']:4d}  V {cc['V']:4d}  A {cc['A']:4d}  mixed {cc['mixed']:4d}  all four present: {all(cc[c] > 0 for c in ['P', 'V', 'A', 'mixed'])}")
allc = np.concatenate([classes(TUN[MAIN][k]) for k in keys])
tuned = allc != "none"
print("\nmain reading, % of tuned neurons:", {c: round(100 * np.mean(allc[tuned] == c), 1) for c in ["P", "V", "A", "mixed"]})
print("main reading, mean % of neurons per field:", {c: round(100 * np.mean([np.mean(classes(TUN[MAIN][k]) == c) for k in keys]), 1) for c in ["P", "V", "A", "mixed", "none"]})

**Result: reproduced.** Under the main reading, which reproduces fig2de-tuned-fraction, all four
classes hold neurons (36 position only, 183 velocity only, 76 acceleration only, 273 mixed), and all
four are present under every reading.

## tuning-consistency

> Tuning was consistent across halves of the data; 95% of neurons had identical tuning to at least
> one feature in the first and last pairs of recordings, and 81% to all three features.

The halves are recordings 1-2 and 3-4. In the six fields whose first movie has no stimulus file
the first half is recording 2 alone.

In [ ]:
def tune_reps(k, reps, rd):
    fd = ssc6.field_data(data, k, reps)
    reg, meth, fam, sign = rd
    f = (ssc6.regress_separate if reg == "sep" else ssc6.regress_joint)(fd["R"], fd["shape"], fd["norm"])
    return ssc6.tuning(f[0], f[1], meth, fam, sign)
for rd in [MAIN, ("joint", "pdep", "all", "both")]:
    one, three, full_t, either, mouse = [], [], [], [], []
    for k in keys:
        s1, s2 = tune_reps(k, {1, 2}, rd), tune_reps(k, {3, 4}, rd)
        same = s1 == s2
        one.append(same.any(1)); three.append(same.all(1)); full_t.append(TUN[rd][k].any(1))
        either.append(s1.any(1) | s2.any(1)); mouse.append(np.full(len(s1), k[0]))
    one, three, full_t, either, mouse = map(np.concatenate, (one, three, full_t, either, mouse))
    print(f"reading {rd}")
    for name, sel in [("all neurons", np.ones_like(one)), ("tuned in the full data", full_t), ("tuned in at least one half", either)]:
        print(f"  {name:28s} n={sel.sum():5d}: identical for >=1 feature {100 * one[sel].mean():5.1f}%, all 3 features {100 * three[sel].mean():5.1f}%")
    if rd == MAIN:
        for m in sorted(by_mouse):
            s = mouse == m
            print(f"    exp{m}: >=1 {100 * one[s].mean():5.1f}%, all 3 {100 * three[s].mean():5.1f}%")
print("expected 95% (85.5 to 104.5) and 81% (72.9 to 89.1)")

**Result: reproduced.** With all neurons as the denominator (the first reading the criterion
lists), the main tuning reading gives 99.5% identical for at least one feature and 86.5% for all
three, both within tolerance. With tuned neurons as the denominator the share identical for all
three features is far lower (printed above), so the match rests on the denominator, which the
paper does not state.

## shuffle-significance

> Scoring tuning by shuffling responses against stimulus amplitude (1000 repeats) in place of the
> regression test gives no qualitative change in the results.

For each neuron and waveform type the responses are permuted across the amplitudes 1000 times;
the p-value is the fraction of shuffled slopes at least as large in magnitude as the observed one.
With the FDR correction (BH over the 3 x N p-values of the field, as the main reading) and
without it (p < 0.05). The SI test is that of fig3g-si-no-spatial-organization. The barrel-map tests
(fig4c, fig4d) cannot be run (blocked below).

In [ ]:
rng_shuf = np.random.default_rng(20170225)
def perm_p(fd, n=1000, rng=rng_shuf):
    R, sh, nm = fd["R"], fd["shape"], fd["norm"]
    p = np.ones((len(R), 3)); b = np.zeros((len(R), 3))
    for j, s in enumerate("PVA"):
        cols = (sh == s) & ~np.isnan(R[0])
        x = nm[cols].astype(float); Y = R[:, cols]
        xc = x - x.mean(); Yc = Y - Y.mean(1, keepdims=True)
        obs = Yc @ xc / (xc @ xc)
        Xp = np.array([rng.permutation(xc) for _ in range(n)])
        sl = (Xp @ Yc.T) / (xc @ xc)
        p[:, j] = (np.abs(sl) >= np.abs(obs)[None]).mean(0); b[:, j] = obs
    return b, p
PERM = {k: perm_p(FD[k]) for k in keys}
SIG_PERM = {"fdr": {k: ssc6.fdr(PERM[k][1]) for k in keys},
            "none": {k: PERM[k][1] < 0.05 for k in keys}}
for name, sig in SIG_PERM.items():
    allc = np.concatenate([classes(sig[k]) for k in keys])
    cc = collections.Counter(allc)
    agree = np.mean(np.concatenate([(sig[k] == TUN[MAIN][k]).all(1) for k in keys]))
    print(f"shuffle test, {name:4s}: non-tuned {100 * cc['none'] / len(allc):.1f}%; P {cc['P']}, V {cc['V']}, A {cc['A']}, mixed {cc['mixed']}; "
          f"neurons with the same tuning as the regression test {100 * agree:.1f}%")

## fig3b-example-correlation (blocked)

The example field of Figure 3A and 3B is not identified (claims file, open detail 11). Figure 3A is
a raster of calcium activity over 90 s, which cannot be matched to fields whose ROIs are found again
and whose tuning colours depend on the analysis. Blocked: metadata insufficient. The claims that
use this field (fig3d-example-correlation-distance, fig3f-example-si-distance) are blocked with it.

## fig3c-correlation-above-shuffle

> Across the data set, pairwise correlations of neurons in a field were higher than for shuffled
> data (0.1004 against 0.011, mean over 49 fields).

Readings (criterion): the signal is the corrected fluorescence (main) or dF/F0; the shuffle permutes
each neuron's stimulus segments (onset to next onset) within each recording or over the whole
concatenated series; the shuffled value is the mean over 100 permutations, and one permutation is
shown as a variant. The recordings that have a stimulus file are concatenated (three in six fields).

In [ ]:
NPERM = 100
CORR = {}
t0 = time.time()
for sig_name in ["corrected", "dff"]:
    for within in [True, False]:
        if sig_name == "dff" and not within:
            continue
        rows = []
        for i, k in enumerate(keys):
            fd = FD[k]; X = fd[sig_name]
            d, _ = ssc6.pair_distances_mm(fd["centroids"])
            c = ssc6.spearman_upper(X)
            rng = np.random.default_rng([303, i, int(within), sig_name == "dff"])
            far = d > 0.2
            shm, shs, shs300, shfar = [], [], [], []
            for p in range(NPERM):
                cs = ssc6.spearman_upper(ssc6.shuffled_series(X, fd["bounds"], rng, within))
                shm.append(cs.mean()); shs.append(ssc6.slope(d, cs)); shs300.append(ssc6.slope(d[d <= 0.3], cs[d <= 0.3]))
                shfar.append(ssc6.slope(d[far], cs[far]))
            rows.append({"key": k, "true": c.mean(), "shuf": np.mean(shm), "shuf1": shm[0], "d": d, "c": c,
                         "slope": ssc6.slope(d, c), "slope_shuf": np.mean(shs),
                         "slope300": ssc6.slope(d[d <= 0.3], c[d <= 0.3]), "slope300_shuf": np.mean(shs300),
                         "slope_far": ssc6.slope(d[far], c[far]), "slope_far_shuf": np.mean(shfar),
                         "n_far": int(far.sum()), "n": len(X)})
        CORR[(sig_name, within)] = rows
print(f"computed in {(time.time() - t0) / 60:.1f} min")
for rd, rows in CORR.items():
    t = np.array([r["true"] for r in rows]); s = np.array([r["shuf"] for r in rows]); s1 = np.array([r["shuf1"] for r in rows])
    w = stats.wilcoxon(t, s)
    print(f"signal {rd[0]:9s} shuffle {'within recording' if rd[1] else 'whole series':16s}: true {t.mean():.4f} +- {stats.sem(t):.4f}, "
          f"shuffled {s.mean():.4f} +- {stats.sem(s):.4f} (one permutation {s1.mean():.4f}); true > shuffled in {(t > s).sum()}/{len(t)} fields; "
          f"Wilcoxon p = {w.pvalue:.2g}")
print("expected true 0.1004 (0.0924 to 0.1084), shuffled 0.011 (0.0086 to 0.0134), P < 0.05")
rows = CORR[("corrected", True)]
for m in sorted(by_mouse):
    sel = [r for r in rows if r["key"][0] == m]
    print(f"  exp{m}: true {np.mean([r['true'] for r in sel]):.3f}, shuffled {np.mean([r['shuf'] for r in sel]):.3f} ({len(sel)} fields)")

**Result: partially reproduced.** Correlations exceed the shuffled ones in all 49 fields under every
reading (Wilcoxon p = 3.6e-15), but no reading gives both means within tolerance. On corrected
fluorescence (the main reading) the true mean is 0.2328, more than twice the paper's 0.1004, and
the shuffled means are 0.1195 (within recordings) and 0.0351 (whole series). On dF/F0 the shuffled
mean, 0.0133, is within tolerance and the true mean, 0.1097, is just above it (0.0924 to 0.1084).

## neuropil-correlation

> Neuropil ROIs were more correlated with each other than neurons were (0.1792 against 0.1004 mean
> correlation; 10 neuropil ROIs per field).

Ten disks of the median somatic ROI area per field, placed at random away from somata and their
halos and outside the darkest quarter of the mean image (vessels), two placements (seeds 0 and 1 in
`ssc6.process_field`). Their traces are the raw mean minus the vessel background (no halo), correlated
over the concatenated recordings as for neurons; the neuronal values are those of the main reading
(corrected fluorescence).

In [ ]:
neur = np.array([r["true"] for r in CORR[("corrected", True)]])
for pl in range(2):
    npc = np.array([ssc6.spearman_upper(FD[k]["neuropil"][pl]).mean() for k in keys])
    nroi_np = [len(FD[k]["neuropil"][pl]) for k in keys]
    w = stats.wilcoxon(npc, neur)
    print(f"placement {pl + 1}: neuropil ROIs per field {min(nroi_np)} to {max(nroi_np)} (total {sum(nroi_np)}); "
          f"neuropil {npc.mean():.4f} +- {stats.sem(npc):.4f} vs neurons {neur.mean():.4f}; neuropil higher in "
          f"{(npc > neur).sum()}/{len(keys)} fields; Wilcoxon p = {w.pvalue:.2g}; median difference {np.median(npc - neur):+.4f}")
print("expected neuropil 0.1792 (0.1695 to 0.1889), higher than neurons, P < 0.05")

**Result: partially reproduced.** In both placements the neuropil ROIs are more correlated with
each other than the neurons are, in all 49 fields (0.6123 and 0.6012 against 0.2328, Wilcoxon p =
3.6e-15), but the neuropil means are more than three times the paper's 0.1792, outside the
tolerance.

## fig3d-example-correlation-distance (blocked)

Depends on fig3b-example-correlation, whose example field is not identified. Blocked: metadata
insufficient.

## fig3e-correlation-falls-with-distance

> Over the data set, correlations fell with distance between neurons, with a median slope of -0.29
> per mm against -0.0062 per mm for shuffled data.

Per field, the least-squares slope of pair correlation on centroid distance (mm), over all pairs and
over pairs up to 300 um (open detail 8), for each signal and shuffle reading of fig3c.

In [ ]:
def msem(x):
    x = np.asarray(x); return np.median(x), 1.2533 * np.std(x, ddof=1) / np.sqrt(len(x))
for rd, rows in CORR.items():
    for lab, a, b in [("all pairs", "slope", "slope_shuf"), ("pairs <= 300 um", "slope300", "slope300_shuf")]:
        t = np.array([r[a] for r in rows]); s = np.array([r[b] for r in rows])
        w = stats.wilcoxon(t, s)
        (mt, et), (ms, es) = msem(t), msem(s)
        print(f"{rd[0]:9s} {'within' if rd[1] else 'whole ':6s} {lab:15s}: median slope {mt:+.3f} +- {et:.3f} /mm, shuffled {ms:+.3f} +- {es:.3f}; "
              f"true below shuffled in {(t < s).sum()}/{len(t)}; Wilcoxon p = {w.pvalue:.2g}")
dd = np.concatenate([r["d"] for r in CORR[("corrected", True)]])
print(f"pair distances: {len(dd)} pairs, {dd.min() * 1000:.0f} to {dd.max() * 1000:.0f} um, {np.mean(dd > 0.3) * 100:.1f}% beyond 300 um")
print("expected median slope -0.29 (-0.352 to -0.228), shuffled -0.0062 (-0.0202 to 0.0078), P < 0.05")

**Result: partially reproduced.** Correlations fall with distance more steeply than in shuffled
data under every reading (true below shuffled in 42 to 48 of 49 fields, Wilcoxon p <= 8.1e-09),
and the true median slope is within tolerance (-0.287 per mm on corrected fluorescence, -0.237 on
dF/F0), but the shuffled median is outside it under every reading (-0.094, -0.024 and -0.021 per
mm, against -0.0202 to 0.0078).

## correlation-gradient-beyond-200um

> For pairs of neurons more than 200 um apart the correlation-distance slope did not differ from
> shuffled data (-0.13 against -0.109 per mm, P = 0.106), so correlations fall off over about 200 um.

Fields with more than 4 pairs over 200 um apart; the slope fitted to those pairs (main reading) and
to all pairs of those fields (variant). Judged on the Wilcoxon signed-rank test, P >= 0.05.

In [ ]:
for rd, rows in CORR.items():
    sel = [r for r in rows if r["n_far"] > 4]
    t_far = np.array([r["slope_far"] for r in sel]); s_far = np.array([r["slope_far_shuf"] for r in sel])
    w = stats.wilcoxon(t_far, s_far)
    t_all = np.array([r["slope"] for r in sel]); s_all = np.array([r["slope_shuf"] for r in sel])
    w2 = stats.wilcoxon(t_all, s_all)
    (mt, et), (ms, es) = msem(t_far), msem(s_far)
    print(f"{rd[0]:9s} {'within' if rd[1] else 'whole '}: fields {len(sel)} of {len(rows)}; far pairs: median {mt:+.3f} +- {et:.3f} /mm, shuffled {ms:+.3f} +- {es:.3f}, "
          f"p = {w.pvalue:.3g}, median difference {np.median(t_far - s_far):+.3f}; all pairs of those fields: p = {w2.pvalue:.2g}, "
          f"median difference {np.median(t_all - s_all):+.3f}")
print("pairs over 200 um per field: median", np.median([r["n_far"] for r in CORR[("corrected", True)]]),
      "range", min(r["n_far"] for r in CORR[("corrected", True)]), "to", max(r["n_far"] for r in CORR[("corrected", True)]))
print("expected: fields 30 (27 to 33), median -0.13 (+-0.17), shuffled -0.109 (+-0.085), P >= 0.05")

**Result: reproduced.** Under the main reading (corrected fluorescence, shuffles within recordings,
slopes over the pairs more than 200 um apart) the slopes do not differ from shuffled (median -0.027
against -0.021 per mm, p = 0.459), and the other two readings agree (p = 0.76 and 0.412). Every one
of the 49 fields has more than 4 such pairs, where the paper has 30 fields, and the shuffled median
lies just outside its tolerance (-0.194 to -0.024). Fitted over all pairs of those fields, the
slopes do differ from shuffled (p <= 3.1e-13).

## fig3f-example-si-distance (blocked)

Depends on fig3b-example-correlation, whose example field is not identified. Blocked: metadata
insufficient.

## fig3g-si-no-spatial-organization

> Pooled across fields, tuning similarity showed no consistent spatial organization; the slope of
> SI on distance did not differ from shuffled data (median -0.467 against -0.0169 per mm, 26 fields,
> P = 0.114), so neurons tuned to different features are interspersed.

Tuning of the main reading. SI is the dot product of the tuning vectors (non-significant
coefficients zero); shuffles reassign the vectors across the neurons of a field, 100 times.
Readings: all pairs of fields with at least two tuned neurons (main); pairs of tuned neurons only;
fields with at least two neurons tuned to a common feature; and SI of unit-length vectors.

In [ ]:
def si_test(sigs, coefs, mode="all", unit=False, seed=0, nperm=100):
    out = []
    for i, k in enumerate(keys):
        sig = sigs[k]; b = coefs[k]
        V = np.where(sig, b, 0.0)
        tuned = sig.any(1)
        if mode == "common":
            if not (sig.sum(0) >= 2).any():
                continue
        elif tuned.sum() < 2:
            continue
        if unit:
            nrm = np.linalg.norm(V, axis=1, keepdims=True); V = np.where(nrm > 0, V / np.where(nrm > 0, nrm, 1), 0)
        cen = FD[k]["centroids"]
        idx = np.flatnonzero(tuned) if mode == "tuned" else np.arange(len(V))
        d, iu = ssc6.pair_distances_mm(cen[idx])
        Vi = V[idx]
        si = (Vi @ Vi.T)[iu]
        rng = np.random.default_rng([seed, i])
        sh = []
        for _ in range(nperm):
            Vp = Vi[rng.permutation(len(Vi))]
            sh.append(ssc6.slope(d, (Vp @ Vp.T)[iu]))
        if len(d) < 2:
            continue
        out.append((k, ssc6.slope(d, si), np.mean(sh), len(idx), len(d)))
    return out
coefs_main = {k: fits[k]["sep"][0] for k in keys}
SI = {}
print("SI scale: raw (coefficients in dF/F0 per amplitude unit, as fractions), percent (dF/F0 in %, SI x 1e4), unit-length vectors")
for mode in ["all", "tuned", "common"]:
    for scale in ["raw", "percent", "unit"]:
        res = si_test(TUN[MAIN], coefs_main, mode, scale == "unit", seed=407)
        SI[(mode, scale)] = res
        f = 1e4 if scale == "percent" else 1.0
        t = f * np.array([r[1] for r in res]); s = f * np.array([r[2] for r in res])
        w = stats.wilcoxon(t, s)
        (mt, et), (ms, es) = msem(t), msem(s)
        print(f"pairs {mode:6s} {scale:7s}: fields {len(res)}, pairs {sum(r[4] for r in res)}; median slope {mt:+.3g} +- {et:.2g} /mm, "
              f"shuffled {ms:+.3g} +- {es:.2g}; p = {w.pvalue:.3g}; median difference {np.median(t - s):+.3g}; true below shuffled in {(t < s).sum()}/{len(t)}")
print("expected: fields 26 (23 to 29), median -0.467 (-0.514 to -0.420), shuffled -0.0169 (+-0.119), P >= 0.05")
res = SI[("all", "raw")]
print("main reading by mouse (fields, median true minus shuffled):",
      {m: (len([r for r in res if r[0][0] == m]), float(f"{np.median([r[1] - r[2] for r in res if r[0][0] == m]):.3g}") if any(r[0][0] == m for r in res) else None) for m in sorted(by_mouse)})

**Result: reproduced.** Under the main reading (all pairs of the 35 fields with at least two tuned
neurons, raw SI) the slopes of SI on distance do not differ from shuffled (p = 0.614), nor under any
variant that could be run (p printed above, all above 0.05). The field count, 35, is outside the
tolerance of 23 to 29, and the median slopes are near zero on the raw scale (coefficients as
fractions); on the percent scale and with unit vectors they are printed above, and none is near
the paper's -0.467 per mm.

## shuffle-significance, continued: the SI test with shuffle-based tuning

In [ ]:
coefs_perm = {k: PERM[k][0] for k in keys}
for name, sig in SIG_PERM.items():
    res = si_test(sig, coefs_perm, "all", False, seed=511)
    t = np.array([r[1] for r in res]); s = np.array([r[2] for r in res])
    w = stats.wilcoxon(t, s)
    print(f"shuffle tuning, {name:4s}: SI test over {len(res)} fields, median slope {np.median(t):+.3g} /mm, shuffled {np.median(s):+.3g}, p = {w.pvalue:.3g}, "
          f"median difference {np.median(t - s):+.3g}")

**Result: reproduced.** With tuning scored by the 1000-shuffle test, non-tuned neurons remain the
majority (77.4% with the FDR correction, 57.0% without), all four classes are present (38, 193, 81
and 282 neurons with the correction; 163, 284, 190 and 492 without), and the SI test still finds no
difference from shuffled (p = 0.521 and 0.629). With the correction 97.1% of neurons have the same
tuning as under the regression test. The barrel-map tests of Figure 4 could not be run.

## fig4c-barrel-septum-tuned and fig4d-border-distance-by-feature (blocked)

Each histology file (experiments 8 to 10) is a single-page figure: the cytochrome-oxidase sections,
the barrel outlines, the in vivo surface image, and one dashed square for the imaged area (rendered
in the data check). It gives neither the position of each 2-photon field within that square nor a
scale, and the paper's own placement used "a single image of the cortical surface taken before
starting the 2-photon acquisition", which is not in the data set. The neurons cannot be put on the
barrel map. Blocked: metadata insufficient; fig4d depends on fig4c.

## Summary

In [ ]:
print(f"notebook run time {(time.time() - t_start) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

| Claim | Outcome |
|---|---|
| dataset-size | partially reproduced |
| stimulus-set | partially reproduced |
| fig2ab-example-neuron | blocked (metadata insufficient) |
| responsive-fraction | not reproduced (rests on an assumed definition) |
| fig2de-tuned-fraction | reproduced |
| fig2e-all-categories | reproduced |
| tuning-consistency | reproduced |
| shuffle-significance | reproduced |
| fig3b-example-correlation | blocked (metadata insufficient) |
| fig3c-correlation-above-shuffle | partially reproduced |
| neuropil-correlation | partially reproduced |
| fig3d-example-correlation-distance | blocked (metadata insufficient) |
| fig3e-correlation-falls-with-distance | partially reproduced |
| correlation-gradient-beyond-200um | reproduced |
| fig3f-example-si-distance | blocked (metadata insufficient) |
| fig3g-si-no-spatial-organization | reproduced |
| fig4c-barrel-septum-tuned | blocked (metadata insufficient) |
| fig4d-border-distance-by-feature | blocked (metadata insufficient) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data set holds raw movies only, so the whole
pipeline was run again: whole-pixel rigid registration, ROIs found by an automatic blob detector in
place of Caltracer with manual supervision (2626 ROIs, against the paper's 1054 neurons), a halo
ring and a vessel background whose sizes the paper does not give, and dF/F0 and responses as the
Methods state. Every downstream number rests on these ROIs; the fractions, correlations and slopes
of the paper were computed on its own, smaller set of cells, whose selection is not described. One ROI
set serves the four movies of a field, aligned by their mean images (shifts of 0 to 2 pixels,
printed in the data check); that the ROIs are the same cells across movies is an assumption, and
tuning-consistency gives the tuning per half of the recordings. The six movies without a stimulus
file are left out of the response and correlation analyses (three recordings in those fields). Details the paper leaves open were run under each reading the claims
file names; the readings are printed in each section.

What was found about the dataset. Ten mice, 49 fields and four movies per field, as the paper
states. Six movies (the first recording of a field in six mice) have no stimulus file. Each
complete stimulus file holds 89 deflections, with the smallest position stimulus presented 9 times,
and 188 of 190 files hold the same deflection order, so the order was not randomised between
recordings. Nine stimulus files start just after the first frame pulse. The description's range of
3 to 6 fields per mouse is 3 to 8 in the data, and the tar's README and the description call the
data set ssc-5. No ROIs, traces, tuning labels or neuron positions are deposited, and the histology
figures do not locate the fields or their neurons on the barrel map, so the analyses of Figure 4
cannot be repeated.